In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.text_cell_render.rendered_html{font size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input{font-family:Consolas; font-size:12pt;}
div.prompt {min width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe {font-size:12px;}
</style>
"""))

# 문장 -> 임베딩 벡터(1차원 숫자 배열)
- openAI API의 키 OPENAI_API_KEY (text-embedding-3-large)를 .env에 추가
- upstage(https://console.upstage.ai) 의 키를 UPSTAGE_API_KEY를 .env에 추가

# 1. 환경변수 load

In [4]:
from dotenv import load_dotenv
import os
load_dotenv()
openai_key = os.getenv('OPENAI_API_KEY')
upstage_key = os.getenv('UPSTAGE_API_KEY')

# 2. 유사도 계산하는 방법
- 1. 유클리드 거리 : 두 벡터간 거리가 가까운지
- 2. 코사인 유사도 : 두 벡터간 방향이 유사한지
- 3. dot product : 두 벡터간 곱을 사용하여 거리와 방향을 모두 고려



In [5]:
import numpy as np
def cosine_similarity(vec1, vec2):
    """두 백터 사이의 코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1) 
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0.0
    return dot_product / (norm_vec1*norm_vec2)


# 3. openAI API의 enbedding model 사용
- text-embedding-3-large

In [6]:
from openai import OpenAI
openai_client = OpenAI()

In [7]:
# text-embedding-3-large
response = openai_client.embeddings.create(
    input = "The king is the prince's father.",
    model = "text-embedding-3-large"
)

In [8]:
response

CreateEmbeddingResponse(data=[Embedding(embedding=[0.029411012306809425, 0.026699358597397804, -0.011381123214960098, 0.020845837891101837, 0.00936041958630085, -0.02716868370771408, 0.02510887011885643, -0.020572064444422722, 3.908492726623081e-05, 0.005113683640956879, 0.03311346471309662, 0.020102739334106445, -0.0485229566693306, 0.013004204258322716, 0.03413033485412598, 0.016413327306509018, -0.055380310863256454, 0.030558250844478607, -0.04004903882741928, -0.009334346279501915, -0.012691321782767773, -0.04059658572077751, -0.0018789282767102122, -0.0070268334820866585, -0.03259199112653732, -0.0001560341625008732, -0.00883242953568697, 0.003813263028860092, -0.019763782620429993, 0.0006905428017489612, -0.0038295588456094265, 0.012358883395791054, 0.027064388617873192, 0.016113480553030968, -0.023844299837946892, -0.019333569332957268, -0.01882513426244259, -0.027142610400915146, -0.009425603784620762, -0.020050592720508575, -0.03290487453341484, -0.0474017933011055, -0.0460720

In [10]:
king_vector = np.array(response.data[0].embedding) # 문자를 숫자로 바꾼 vector
print(king_vector.shape)

(3072,)


In [12]:
queen_response = openai_client.embeddings.create(
    input = "Th queen is the prince's mother",
    model = "text-embedding-3-large"
)

In [14]:
queen_vector = np.array(queen_response.data[0].embedding)
print(queen_vector.shape)

(3072,)


In [16]:
# king_vector와 queen_vector의 유사도
king_queen_similarity = cosine_similarity(king_vector, queen_vector)
print('king과 queen의 유사도 :', king_queen_similarity)

king과 queen의 유사도 : 0.6261620466552995


In [17]:
slave_response = openai_client.embeddings.create(
    input = "The slave begs",
    model = "text-embedding-3-large"
)

In [19]:
slave_vector = np.array(slave_response.data[0].embedding)
slave_vector.shape

(3072,)

In [20]:
# queen_vector와 slave_vector의 유사도
queen_slave_similarity = cosine_similarity(slave_vector, queen_vector)
print('queen과 slave의 유사도 :', queen_slave_similarity)

king과 slave의 유사도 : 0.15738522561936533


In [21]:
# 한국어 문장을 벡터로 바꿔도 유사도가 비슷한지 확인

In [22]:
response_king_kor = openai_client.embeddings.create(
    input = '왕은 왕자의 아버지다',
    model = 'text-embedding-3-large'
)
kor_king_vector = np.array(response_king_kor.data[0].embedding)

In [23]:
kor_king_vector.shape

(3072,)

In [24]:
response_queen_kor = openai_client.embeddings.create(
    input = '여왕은 왕자의 어머니다',
    model = 'text-embedding-3-large'
)
kor_queen_vector = np.array(response_queen_kor.data[0].embedding)
kor_queen_vector.shape

(3072,)

In [26]:
print('왕 문장과 여왕 문장의 유사도 :', cosine_similarity(kor_queen_vector, kor_king_vector))

왕 문장과 여왕 문장의 유사도 : 0.5744286874279486


In [27]:
response_slave_kor = openai_client.embeddings.create(
    input = '노예가 구걸한다',
    model = 'text-embedding-3-large'
)
kor_slave_vector = np.array(response_slave_kor.data[0].embedding)

In [28]:
print('여왕 문장과 노예 문장의 유사도 :', cosine_similarity(kor_queen_vector, kor_slave_vector))

여왕 문장과 노예 문장의 유사도 : 0.19036176182157677


In [29]:
# queen vector와 kor_queen_vector의 유사도 (의미가 동일한 두 문자)
print('queen 문장과 여왕 문장의 유사도 :', cosine_similarity(queen_vector, kor_queen_vector))

queen 문장과 여왕 문장의 유사도 : 0.5870326362965361


# 4. upstage의 embedding model 사용
- 한국어에서는 openai 보다 성능이 좋다

In [30]:
upstage_client = OpenAI(
    api_key=upstage_key,
    base_url="https://api.upstage.ai/v1"
)

In [31]:
response = upstage_client.embeddings.create(
    input="The king is prince's father",
    model="embedding-query"
)

In [34]:
up_king_vector = np.array(response.data[0].embedding)
print(up_king_vector.shape)

(4096,)


In [35]:
response = upstage_client.embeddings.create(
    input="The queen is prince's mother",
    model="embedding-query"
)
up_queen_vector = np.array(response.data[0].embedding)
print(up_queen_vector.shape)

(4096,)


In [36]:
print('king 문장과 queen 문장의 유사도 :', cosine_similarity(up_king_vector, up_queen_vector))

king 문장과 queen 문장의 유사도 : 0.6695324547493329


In [37]:
response = upstage_client.embeddings.create(
    input="왕은 왕자의 아버지다",
    model="embedding-query"
)
up_kor_king_vector = np.array(response.data[0].embedding)
print(up_kor_king_vector.shape)

(4096,)


In [38]:
response = upstage_client.embeddings.create(
    input="여왕은 왕자의 어머니다",
    model="embedding-query"
)
up_kor_queen_vector = np.array(response.data[0].embedding)
print(up_kor_queen_vector.shape)

(4096,)


In [40]:
print('왕 문장과 여왕 문장의 유사도 :', cosine_similarity(up_kor_king_vector, up_kor_queen_vector))

왕 문장과 여왕 문장의 유사도 : 0.6502680711224584


In [42]:
print('king 문장과 왕 문장의 유사도 :', cosine_similarity(up_king_vector, up_kor_king_vector))

king 문장과 왕 문장의 유사도 : 0.8495946530745893


In [43]:
print('queen 문장과 여왕 문장의 유사도 :', cosine_similarity(up_queen_vector, up_kor_queen_vector))

queen 문장과 여왕 문장의 유사도 : 0.8021974162769381
